### Random Forest:
1) запуск алгоритма с параметрами по умолчанию и вывод некоторой статистики
2) запуск optuna-оптимизации по части гиперпараметров
3) визуализация optuna: важность параметров и контуры
4) запуск алгоритма с найденными гиперпараметрами и вывод предварительной статистики
5) сохраняем результаты дефолтного и оптимизированного алгоритма

In [1]:
from private.utils import get_reduced_mnist_data, memory_check
from public.classification_utils import RAN_FOR
from public.models import ClassificationProcessor

with memory_check():
    df = get_reduced_mnist_data()
    processor = ClassificationProcessor(df, "label")   
    processor.calculate({
        RAN_FOR : {},
    })
    processor.report(RAN_FOR)
    processor.pick_model(RAN_FOR)

random_forest took 36.582 seconds

	random_forest


pr_auc,roc_auc,accuracy
0.992922,0.998945,0.966929


,precision,recall,f1-score,support
0,0.974,0.992,0.983,1381.000
1,0.984,0.983,0.983,1575.000
2,0.964,0.964,0.964,1398.000
3,0.962,0.956,0.959,1428.000
4,0.970,0.962,0.966,1365.000
5,0.966,0.956,0.961,1263.000
6,0.970,0.982,0.976,1375.000
7,0.973,0.968,0.971,1459.000
8,0.962,0.958,0.960,1365.000
9,0.942,0.946,0.944,1391.000


Memory Increased by: 86.30 MB


#### запуск optuna-оптимизации по части гиперпараметров

In [2]:
import optuna

from public.optuna_utils import OPT_RAN_FOR, optimize

with memory_check():
    study = optimize(
        model_type=OPT_RAN_FOR, 
        df=df, 
        target_column="label",
        n_trials=20,
        log_level=optuna.logging.INFO
    )
    print(f"Наилучшие значения гиперпараметров {study.best_params}")
    print(f"pr_auc на обучающем наборе: {study.best_value:.2f}")

[I 2026-08-17 01:58:56,823] A new study created in memory with name: Random Forest


  0%|          | 0/20 [00:00<?, ?it/s]

optuna_optimize took 127.795 seconds
[I 2026-08-17 02:01:04,839] Trial 4 finished with value: 0.8282997123825263 and parameters: {'n_estimators': 140, 'max_depth': 3, 'min_samples_leaf': 5}. Best is trial 4 with value: 0.8282997123825263.
optuna_optimize took 161.794 seconds
[I 2026-08-17 02:01:38,790] Trial 9 finished with value: 0.8270304723526426 and parameters: {'n_estimators': 180, 'max_depth': 3, 'min_samples_leaf': 5}. Best is trial 4 with value: 0.8282997123825263.
optuna_optimize took 196.510 seconds
[I 2026-08-17 02:02:13,498] Trial 5 finished with value: 0.8866241160164732 and parameters: {'n_estimators': 170, 'max_depth': 4, 'min_samples_leaf': 5}. Best is trial 5 with value: 0.8866241160164732.
optuna_optimize took 211.809 seconds
[I 2026-08-17 02:02:28,801] Trial 11 finished with value: 0.9222672042831102 and parameters: {'n_estimators': 140, 'max_depth': 5, 'min_samples_leaf': 3}. Best is trial 11 with value: 0.9222672042831102.
optuna_optimize took 217.452 seconds
[I 20

#### Визуализация optuna:
1) Сравнение важности гиперпараметров
2) Отрисовка контура оптимизации. Помогает выбрать направление дальнейшей оптимизации в сторону "темных" областей

In [3]:
from optuna.visualization import plot_param_importances

plot_param_importances(study)

![Project Screenshot](./data/optuna/ran_for_feature_importance.png)

In [4]:
from optuna.visualization import plot_contour

plot_contour(study)

![Project Screenshot](./data/optuna/ran_for_contour.png)

#### Применение найденных лучших гиперпараметров:

In [7]:
with memory_check():
    alter_title = f'{RAN_FOR}_tuned'
    processor.calculate({
        RAN_FOR : {
            "n_estimators": 100,
            "min_samples_leaf": 3,
            "max_depth": 10,
            'alter_title': alter_title
        },
    })
    processor.report(alter_title)
    processor.pick_model(alter_title)

random_forest took 22.515 seconds

	random_forest_tuned


pr_auc,roc_auc,accuracy
0.983013,0.997226,0.943929


,precision,recall,f1-score,support
0,0.964,0.983,0.974,1381.000
1,0.959,0.977,0.968,1575.000
2,0.947,0.938,0.943,1398.000
3,0.940,0.929,0.934,1428.000
4,0.946,0.924,0.935,1365.000
5,0.957,0.928,0.942,1263.000
6,0.955,0.972,0.963,1375.000
7,0.951,0.942,0.947,1459.000
8,0.934,0.919,0.927,1365.000
9,0.886,0.921,0.903,1391.000


Memory Increased by: -72.18 MB


#### Мини-репорт:

In [6]:
dec_tr = next((model for model in processor.models if model.title == RAN_FOR), None)
dec_tr_tuned = next((model for model in processor.models if model.title == alter_title), None)

print(f"{RAN_FOR} : {alter_title} >> {dec_tr.pr_auc} : {dec_tr_tuned.pr_auc}")

random_forest : random_forest_tuned >> 0.9929221454605524 : 0.983012829765132
